# Colors

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get seaborn muted palette
palette = sns.color_palette("muted")

# Optional: custom names (you can change these)
names = [
    "blue", "orange", "green", "red",
    "purple", "brown", "pink", "gray", "olive", "cyan"
]

def rgb_to_255(rgb):
    return tuple(int(round(c * 255)) for c in rgb)

def rgb_to_hex(rgb):
    return ''.join(f"{c:02X}" for c in rgb)

print("Seaborn 'muted' palette:\n")

latex_lines = []

for i, color in enumerate(palette):
    rgb_255 = rgb_to_255(color)
    hex_val = rgb_to_hex(rgb_255)

    name = names[i] if i < len(names) else f"color{i}"

    print(f"{name}:")
    print(f"  RGB (0-255): {rgb_255}")
    print(f"  HEX: #{hex_val}")
    print()

    latex = f"\\definecolor{{sb{name}}}{{HTML}}{{{hex_val}}}"
    latex_lines.append(latex)

# Print LaTeX definitions
print("LaTeX definitions:\n")
for line in latex_lines:
    print(line)

# Plot the colors
fig, ax = plt.subplots(figsize=(8, 2))

for i, color in enumerate(palette):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
    label = names[i] if i < len(names) else str(i)
    ax.text(i + 0.5, -0.2, label, ha='center', va='top', fontsize=10)

ax.set_xlim(0, len(palette))
ax.set_ylim(0, 1)
ax.axis('off')

plt.show()

# Temporal Self-Correcting Point Process

In [ ]:
#!/usr/bin/env python3
"""
Self-Correcting Temporal Point Process with Brownian Noise – *Extended v3.2*
============================================================================
Generates two separate plots:

  • **Original trajectory** with the chosen event highlighted.
  • **Counterfactual trajectory** where that event is vetoed by forcing its
    uniform sample to 1.0 (no reset), **plus a dashed vertical line marking
    where that event would have occurred**.

All plots share identical proportions by fixing the y-axis range to [0, 3.3].
The event to suppress is controlled by the hyper-parameter
`SUPPRESS_EVENT_NO` (default 5).
"""
# ---------------------------------------------------------------------
# Imports & styling
# ---------------------------------------------------------------------
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white", palette="muted", rc={
    "axes.titlesize": 28,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
})
PALETTE = sns.color_palette("muted")

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
OUTPUT_FOLDER      = Path("self_correcting_TPP")
CSV_ORIG           = OUTPUT_FOLDER / "self_correcting_path.csv"
CSV_CF             = OUTPUT_FOLDER / "counterfactual_path.csv"

TIME_HORIZON       = 10.0
NUM_STEPS          = 1000          # Δt = 0.01
DT                 = TIME_HORIZON / NUM_STEPS
DRIFT              = 1.0
SIGMA              = 0.25
SEED               = 42

SUPPRESS_EVENT_NO  = 3             # ← hyper-parameter: which event to veto

# Common y-axis range for every plot
YLIM = (-0.02, 3.4)

# --- ONLY CHANGE: colors ---
BLUE  = "#4878D0"
RED   = "#D65F5F"
GREEN = "#6ACC64"

# ---------------------------------------------------------------------
# Helper: simulate given uniform + Brownian noise
# ---------------------------------------------------------------------
def simulate_path(uniform: np.ndarray, brown_inc: np.ndarray):
    """Return intensity, cumulative events, event times, index of every event."""
    intensity = np.zeros(NUM_STEPS + 1)
    events_N  = np.zeros(NUM_STEPS + 1, dtype=int)
    event_ts: list[float] = []
    event_indices: list[int] = []

    for i in range(1, NUM_STEPS + 1):
        event = 1 if uniform[i] < intensity[i-1] * DT else 0
        events_N[i] = events_N[i-1] + event

        if event:
            intensity[i] = 0.0
            event_ts.append(times[i])
            event_indices.append(i)
        else:
            intensity[i] = max(0.0, intensity[i-1] + DRIFT * DT + brown_inc[i])

    return intensity, events_N, event_ts, event_indices

# ---------------------------------------------------------------------
# Generate shared noise
# ---------------------------------------------------------------------
np.random.seed(SEED)

times = np.linspace(0.0, TIME_HORIZON, NUM_STEPS + 1)

uniform_base = np.random.uniform(0.0, 1.0, NUM_STEPS + 1)
brown_inc    = np.random.normal(0.0, SIGMA * np.sqrt(DT), NUM_STEPS + 1)
brown_inc[0] = 0.0
brown_path   = np.cumsum(brown_inc)

# ---------------------------------------------------------------------
# Original trajectory
# ---------------------------------------------------------------------
intensity_orig, events_orig, ev_times_orig, ev_idx_list = simulate_path(uniform_base.copy(), brown_inc)

if len(ev_idx_list) < SUPPRESS_EVENT_NO:
    raise RuntimeError(f"Less than {SUPPRESS_EVENT_NO} events generated – adjust parameters.")

idx_suppr = ev_idx_list[SUPPRESS_EVENT_NO - 1]  # zero-based list, 1-based event count
u_suppr   = uniform_base[idx_suppr]
lam_prev  = intensity_orig[idx_suppr - 1]

# ---------------------------------------------------------------------
# Counterfactual trajectory (veto chosen event)
# ---------------------------------------------------------------------
uniform_cf           = uniform_base.copy()
uniform_cf[idx_suppr] = 1.0  # force event to fail
intensity_cf, events_cf, ev_times_cf, _ = simulate_path(uniform_cf, brown_inc)

# ---------------------------------------------------------------------
# Save CSVs
# ---------------------------------------------------------------------
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "time": times,
    "uniform_u": uniform_base,
    "brownian_B": brown_path,
    "intensity": intensity_orig,
    "events_N": events_orig,
}).to_csv(CSV_ORIG, index=False)

pd.DataFrame({
    "time": times,
    "uniform_u": uniform_cf,
    "brownian_B": brown_path,
    "intensity": intensity_cf,
    "events_N": events_cf,
}).to_csv(CSV_CF, index=False)

print(f"Original path → {CSV_ORIG}\nCounterfactual path → {CSV_CF}")



# ---------------------------------------------------------------------
# Figure 1b – Original path (zoom on suppressed event)
# ---------------------------------------------------------------------
fig1b, ax1b = plt.subplots(figsize=(9, 5), dpi=300)
ax1b.plot(times, intensity_orig, color=BLUE, linewidth=2, label="Intensity $\\lambda_t$")
ax1b.scatter(ev_times_orig, np.zeros_like(ev_times_orig)+0.05, marker="o", s=250, color=RED, label="Events")

ax1b.set_xlabel("Time")
ax1b.set_ylabel("Intensity")
ax1b.set_title("Original Self-Correcting TPP")
ax1b.set_ylim(*YLIM)
ax1b.legend(fontsize=16, frameon=False)

sns.despine(ax=ax1b)
fig1b.tight_layout()
fig1b.savefig(OUTPUT_FOLDER / "original_TPP_zoom.jpg", bbox_inches="tight")
fig1b.savefig(OUTPUT_FOLDER / "original_TPP_zoom.pdf", bbox_inches="tight")

# ---------------------------------------------------------------------
# Figure 2 – Counterfactual trajectory
# ---------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(9, 5), dpi=300)
ax2.plot(times, intensity_cf, color=GREEN, linewidth=2, label="Intensity $\\lambda_t$ (cf)")
ax2.scatter(ev_times_cf, np.zeros_like(ev_times_cf)+0.12, marker="^", s=450, color=RED, label="Events (cf)")

# dashed vertical line marking suppressed event
ax2.vlines(
    x=times[idx_suppr],
    ymin=0.0, ymax=2.2,
    linestyle="--",
    linewidth=2,
    color="black",
    alpha=0.8,
    label=f"Suppressed {SUPPRESS_EVENT_NO}-rd event"
)

ax2.set_xlabel("Time")
ax2.set_ylabel("Intensity")
ax2.set_title("Counterfactual Trajectory")
ax2.set_ylim(*YLIM)
ax2.legend(fontsize=16, frameon=False)

sns.despine(ax=ax2)
fig2.tight_layout()
fig2.savefig(OUTPUT_FOLDER / "counterfactual_TPP.jpg", bbox_inches="tight")
fig2.savefig(OUTPUT_FOLDER / "counterfactual_TPP.pdf", bbox_inches="tight")

print("Figures saved in", OUTPUT_FOLDER)

# ---------------------------------------------------------------------
# Figure 3 – Combined side-by-side (original | counterfactual)
# ---------------------------------------------------------------------
fig3, (ax3l, ax3r) = plt.subplots(1, 2, figsize=(18, 5), dpi=300, sharey=True)

# --- Left panel: original ---
ax3l.plot(times, intensity_orig, color=BLUE, linewidth=2, label="Intensity $\\lambda_t$")
ax3l.scatter(ev_times_orig, np.zeros_like(ev_times_orig)+0.05, marker="o", s=250, color=RED, label="Events")
ax3l.set_xlabel("Time")
ax3l.set_ylabel("Intensity")
ax3l.set_title("Original Self-Correcting TPP")
ax3l.set_ylim(*YLIM)
ax3l.legend(fontsize=16, frameon=False)
sns.despine(ax=ax3l)

# --- Right panel: counterfactual (no y-axis label) ---
ax3r.plot(times, intensity_cf, color=GREEN, linewidth=2, label="Intensity $\\lambda_t$ (cf)")
ax3r.scatter(ev_times_cf, np.zeros_like(ev_times_cf)+0.12, marker="^", s=450, color=RED, label="Events (cf)")
ax3r.vlines(
    x=times[idx_suppr],
    ymin=0.0, ymax=2.2,
    linestyle="--",
    linewidth=2,
    color="black",
    alpha=0.8,
    label=f"Suppressed {SUPPRESS_EVENT_NO}-rd event"
)
ax3r.set_xlabel("Time")
# no y-axis label on the right panel
ax3r.set_title("Counterfactual Trajectory")
ax3r.set_ylim(*YLIM)
ax3r.legend(fontsize=16, frameon=False)
sns.despine(ax=ax3r)

fig3.tight_layout()
fig3.subplots_adjust(wspace=0.12)

fig3.savefig(OUTPUT_FOLDER / "combined_TPP.jpg", bbox_inches="tight")
fig3.savefig(OUTPUT_FOLDER / "combined_TPP.pdf", bbox_inches="tight")

print("Combined figure saved as combined_TPP.jpg / combined_TPP.pdf")

plt.show()

# ODE

In [ ]:
# predator_prey_jump.py
#
# Colab-ready alternative to the original script.
# ├─ Folder   : ode_example_jump
# ├─ CSVs     : original_solution.csv / modified_solution.csv   (unchanged)
# └─ Figures  : predator_prey_ode.jpg / predator_prey_ode.pdf   (unchanged)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import solve_ivp

BLUE  = "#4878D0"
RED   = "#D65F5F"


# ---------- I/O helpers ------------------------------------------------------
FOLDER = 'ode_example_jump'        # <─ the only pathname that changed
ORIG_CSV = f'{FOLDER}/original_solution.csv'
MOD_CSV  = f'{FOLDER}/modified_solution.csv'

def create_folder(path: str = FOLDER) -> None:
    os.makedirs(path, exist_ok=True)

# ---------- ODE system -------------------------------------------------------
def predator_prey_ode(t, z, alpha, beta, delta, gamma):
    x, y = z
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

# ---------- Solvers ----------------------------------------------------------
def solve_standard(init_vals, params, num_steps=101, t_end=10.0, csv_path=ORIG_CSV):
    """Unmodified (original) integration."""
    t_eval = np.linspace(0, t_end, num_steps)
    sol = solve_ivp(
        predator_prey_ode,
        [0, t_end],
        init_vals,
        t_eval=t_eval,
        args=tuple(params)
    )
    df = pd.DataFrame({'time': sol.t, 'prey': sol.y[0], 'predator': sol.y[1]})
    df.to_csv(csv_path, index=False)
    print(f"Saved original solution → {csv_path}")
    return df

def solve_with_predator_jump(init_vals, params,
                             jump_val=15, jump_time=5.0,
                             num_steps=101, t_end=10.0, csv_path=MOD_CSV):
    """
    Integrate in two segments:
      0 → jump_time     : normal dynamics
      jump_time → t_end : start from (prey, predator + jump_val)
    """
    # ---- Segment 1 ----------------------------------------------------------
    seg1_steps = int(np.floor(num_steps * jump_time / t_end))
    t_eval1 = np.linspace(0, jump_time, seg1_steps + 1)
    sol1 = solve_ivp(predator_prey_ode, [0, jump_time], init_vals,
                     t_eval=t_eval1, args=tuple(params))

    # ---- Apply instantaneous predator boost --------------------------------
    prey_last, pred_last = sol1.y[:, -1]
    init_seg2 = [prey_last, pred_last + jump_val]

    # ---- Segment 2 ----------------------------------------------------------
    t_eval2 = np.linspace(jump_time, t_end, num_steps - seg1_steps + 1)
    sol2 = solve_ivp(predator_prey_ode, [jump_time, t_end], init_seg2,
                     t_eval=t_eval2, args=tuple(params))

    # ---- Concatenate (avoid duplicate jump_time point from seg2) -----------
    time   = np.concatenate([sol1.t,   sol2.t[1:]])
    prey   = np.concatenate([sol1.y[0], sol2.y[0][1:]])
    pred   = np.concatenate([sol1.y[1], sol2.y[1][1:]])

    df = pd.DataFrame({'time': time, 'prey': prey, 'predator': pred})
    df.to_csv(csv_path, index=False)
    print(f"Saved modified solution (predator +{jump_val} at t={jump_time}) → {csv_path}")
    return df

# ---------- Visualisation ----------------------------------------------------
def visualize_solutions(df1, df2, linestyles=('-', '--'), output_folder=FOLDER):
    sns.set_theme(style="white", palette="muted")
    palette = sns.color_palette("muted")

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

    # Original
    ax.plot(df1['time'], df1['prey'],
            label='Prey (original)',   color=palette[0],
            linestyle=linestyles[0], linewidth=3, alpha=0.7)
    ax.plot(df1['time'], df1['predator'],
            label='Predator (original)', color=palette[3],
            linestyle=linestyles[0], linewidth=3, alpha=0.7)

    # Modified
    ax.plot(df2['time'], df2['prey'],
            label='Prey (cf)',   color=palette[0],
            linestyle=linestyles[1], linewidth=8, alpha=0.4)
    ax.plot(df2['time'], df2['predator'],
            label='Predator (cf)', color=palette[3],
            linestyle=linestyles[1], linewidth=8, alpha=0.4)

    # Title unchanged
    ax.set_title("Predator-Prey ODE", fontsize=28)
    ax.set_xlabel("Time", fontsize=24)
    ax.set_ylabel("Population", fontsize=24)
    ax.tick_params(axis='both', which='major', labelsize=20)
    sns.despine()

    # Legend outside plot
    fig.legend(loc='upper left', bbox_to_anchor=(0.95, 0.95),
               fontsize=18, frameon=False)

    fig.tight_layout()

    jpg_path = os.path.join(output_folder, 'predator_prey_ode.jpg')
    pdf_path = os.path.join(output_folder, 'predator_prey_ode.pdf')
    fig.savefig(jpg_path, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.close(fig)

    print(f"Figure saved as:\n  • {jpg_path}\n  • {pdf_path}")

# ---------- Main -------------------------------------------------------------
if __name__ == "__main__":
    create_folder()

    # Parameters (unchanged)
    init_vals = [10, 5]                 # prey0, predator0
    params    = [1.0, 0.1, 0.075, 1.5]  # α, β, δ, γ

    # Original run
    df_orig = solve_standard(init_vals, params)

    # Modified run: +5 predators at t = 5
    df_mod  = solve_with_predator_jump(init_vals, params, jump_val=10, jump_time=5.0)

    # Visualise both
    visualize_solutions(df_orig, df_mod)


In [ ]:
# predator_prey_jump.py
#
# Colab-ready alternative to the original script.
# ├─ Folder   : ode_example_jump
# ├─ CSVs     : original_solution.csv / modified_solution.csv   (unchanged)
# └─ Figures  : predator_prey_ode.jpg / predator_prey_ode.pdf   (unchanged)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import solve_ivp

BLUE  = "#4878D0"
RED   = "#D65F5F"


# ---------- I/O helpers ------------------------------------------------------
FOLDER = 'ode_example_jump'        # <─ the only pathname that changed
ORIG_CSV = f'{FOLDER}/original_solution.csv'
MOD_CSV  = f'{FOLDER}/modified_solution.csv'

def create_folder(path: str = FOLDER) -> None:
    os.makedirs(path, exist_ok=True)

# ---------- ODE system -------------------------------------------------------
def predator_prey_ode(t, z, alpha, beta, delta, gamma):
    x, y = z
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

# ---------- Solvers ----------------------------------------------------------
def solve_standard(init_vals, params, num_steps=101, t_end=10.0, csv_path=ORIG_CSV):
    """Unmodified (original) integration."""
    t_eval = np.linspace(0, t_end, num_steps)
    sol = solve_ivp(
        predator_prey_ode,
        [0, t_end],
        init_vals,
        t_eval=t_eval,
        args=tuple(params)
    )
    df = pd.DataFrame({'time': sol.t, 'prey': sol.y[0], 'predator': sol.y[1]})
    df.to_csv(csv_path, index=False)
    print(f"Saved original solution → {csv_path}")
    return df

def solve_with_predator_jump(init_vals, params,
                             jump_val=15, jump_time=5.0,
                             num_steps=101, t_end=10.0, csv_path=MOD_CSV):
    """
    Integrate in two segments:
      0 → jump_time     : normal dynamics
      jump_time → t_end : start from (prey, predator + jump_val)
    """
    # ---- Segment 1 ----------------------------------------------------------
    seg1_steps = int(np.floor(num_steps * jump_time / t_end))
    t_eval1 = np.linspace(0, jump_time, seg1_steps + 1)
    sol1 = solve_ivp(predator_prey_ode, [0, jump_time], init_vals,
                     t_eval=t_eval1, args=tuple(params))

    # ---- Apply instantaneous predator boost --------------------------------
    prey_last, pred_last = sol1.y[:, -1]
    init_seg2 = [prey_last, pred_last + jump_val]

    # ---- Segment 2 ----------------------------------------------------------
    t_eval2 = np.linspace(jump_time, t_end, num_steps - seg1_steps + 1)
    sol2 = solve_ivp(predator_prey_ode, [jump_time, t_end], init_seg2,
                     t_eval=t_eval2, args=tuple(params))

    # ---- Concatenate (avoid duplicate jump_time point from seg2) -----------
    time   = np.concatenate([sol1.t,   sol2.t[1:]])
    prey   = np.concatenate([sol1.y[0], sol2.y[0][1:]])
    pred   = np.concatenate([sol1.y[1], sol2.y[1][1:]])

    df = pd.DataFrame({'time': time, 'prey': prey, 'predator': pred})
    df.to_csv(csv_path, index=False)
    print(f"Saved modified solution (predator +{jump_val} at t={jump_time}) → {csv_path}")
    return df

# ---------- Visualisation ----------------------------------------------------
def visualize_solutions(df1, df2, jump_time=5.0, linestyles=('-', '--'), output_folder=FOLDER):
    sns.set_theme(style="white", palette="muted")
    palette = sns.color_palette("muted")

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

    # Original
    ax.plot(df1['time'], df1['prey'],
            label='Prey (original)',   color=palette[0],
            linestyle=linestyles[0], linewidth=3, alpha=0.7)
    ax.plot(df1['time'], df1['predator'],
            label='Predator (original)', color=palette[3],
            linestyle=linestyles[0], linewidth=3, alpha=0.7)

    # Modified: plot the two ODE segments separately so matplotlib never draws
    # a connecting line across the instantaneous predator jump.
    seg_a = df2[df2['time'] <= jump_time]
    seg_b = df2[df2['time'] > jump_time]

    ax.plot(seg_a['time'], seg_a['prey'],
            label='Prey (cf)',   color=palette[0],
            linestyle=linestyles[1], linewidth=7, alpha=0.4)
    ax.plot(seg_b['time'], seg_b['prey'],
            color=palette[0],
            linestyle=linestyles[1], linewidth=7, alpha=0.4)
    ax.plot(seg_a['time'], seg_a['predator'],
            label='Predator (cf)', color=palette[3],
            linestyle=linestyles[1], linewidth=7, alpha=0.4)
    ax.plot(seg_b['time'], seg_b['predator'],
            color=palette[3],
            linestyle=linestyles[1], linewidth=7, alpha=0.4)

    # Title unchanged
    ax.set_title("Predator-Prey ODE", fontsize=28)
    ax.set_xlabel("Time", fontsize=24)
    ax.set_ylabel("Population", fontsize=24)
    ax.tick_params(axis='both', which='major', labelsize=20)
    sns.despine()

    # Legend outside plot
    fig.legend(loc='upper left', bbox_to_anchor=(0.95, 0.95),
               fontsize=18, frameon=False)

    fig.tight_layout()

    jpg_path = os.path.join(output_folder, 'predator_prey_ode.jpg')
    pdf_path = os.path.join(output_folder, 'predator_prey_ode.pdf')
    fig.savefig(jpg_path, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.close(fig)

    print(f"Figure saved as:\n  • {jpg_path}\n  • {pdf_path}")

# ---------- Main -------------------------------------------------------------
if __name__ == "__main__":
    create_folder()

    # Parameters (unchanged)
    init_vals = [10, 5]                 # prey0, predator0
    params    = [1.0, 0.1, 0.075, 1.5]  # α, β, δ, γ

    # Original run
    df_orig = solve_standard(init_vals, params)

    # Modified run: +5 predators at t = 5
    df_mod  = solve_with_predator_jump(init_vals, params, jump_val=10, jump_time=5.0)

    # Visualise both
    visualize_solutions(df_orig, df_mod, jump_time=5.0)

# Brownian Motion

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
DEFAULT_RESOLUTIONS = [
    np.linspace(0, 10, 5),
    np.linspace(0, 10, 11),
    np.linspace(0, 10, 21),
    np.linspace(0, 10, 101)
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def create_folder(folder_path: str = "brownian_motion") -> None:
    """Create the output folder if it does not exist."""
    os.makedirs(folder_path, exist_ok=True)


def generate_and_save_brownian_motion(
    num_paths: int = 4,
    resolutions=None,
    output_folder: str = "brownian_motion",
    seed: int = 42
) -> None:
    """Generate Brownian-motion sample paths and save them as CSV files."""
    np.random.seed(seed)

    if resolutions is None:
        resolutions = DEFAULT_RESOLUTIONS

    for i, t_grid in enumerate(resolutions):
        dt = np.diff(t_grid, prepend=0)
        for path_id in range(num_paths):
            increments = np.random.normal(0, np.sqrt(dt[1:]))
            path = np.concatenate([[0], np.cumsum(increments)])

            df = pd.DataFrame({"time": t_grid, "value": path})
            filename = f"bm_path_{path_id+1}_res_{len(t_grid)}.csv"
            df.to_csv(os.path.join(output_folder, filename), index=False)
            print(f"Saved Brownian-motion path to {filename}")


def visualize_brownian_paths(
    csv_files,
    output_folder: str = "brownian_motion"
) -> None:
    """
    Plot the sampled Brownian-motion paths.  Each legend entry shares the
    colour of its corresponding path because the first segment of every
    path carries a unique *label* that Matplotlib can collect.
    """
    sns.set_theme(style="white", palette="muted")
    palette = [
        "#4878D0",  # blue
        "#6ACC64",  # green
        "#D65F5F",  # red
        "#8C613C",  # brown
    ]

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

    for idx, file in enumerate(csv_files):
        df = pd.read_csv(file)
        time = df["time"]
        value = df["value"]

        colour = palette[idx % len(palette)]
        label = f"Path {idx + 1} ({len(time)} steps)"

        # --- first horizontal segment (with label) ---------------------
        ax.hlines(
            value[0], time[0], time[1],
            color=colour, linestyle="-", linewidth=3, alpha=0.5,
            label=label                    # <- legend handle comes from here
        )

        # --- remainder of the staircase -------------------------------
        for j in range(1, len(time)):
            # vertical jump at t_j
            ax.vlines(
                time[j], value[j - 1], value[j],
                color=colour, linestyle="-", linewidth=3, alpha=0.5
            )
            # horizontal segment (skip the one we already drew)
            if j > 1:
                ax.hlines(
                    value[j - 1], time[j - 1], time[j],
                    color=colour, linestyle="-", linewidth=3, alpha=0.5
                )

    # -----------------------------------------------------------------
    # Cosmetics
    # -----------------------------------------------------------------
    ax.set_title("Brownian Motion Paths", fontsize=28)
    ax.set_xlabel("Time", fontsize=24)
    ax.set_ylabel("Value", fontsize=24)
    ax.tick_params(axis="both", labelsize=20)
    sns.despine()

    # Matplotlib assembles handles/labels automatically because we supplied
    # a label on exactly one segment per path.
    fig.legend(
        loc="upper left",
        bbox_to_anchor=(0.95, 0.95),
        fontsize=18,
        frameon=False
    )

    plt.grid(False)
    fig.tight_layout()

    # -----------------------------------------------------------------
    # Save outputs
    # -----------------------------------------------------------------
    jpg_path = os.path.join(output_folder, "brownian_motion.jpg")
    pdf_path = os.path.join(output_folder, "brownian_motion.pdf")
    fig.savefig(jpg_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved figure as {jpg_path} and {pdf_path}")


# ---------------------------------------------------------------------
# Main program
# ---------------------------------------------------------------------
if __name__ == "__main__":
    create_folder("brownian_motion")

    generate_and_save_brownian_motion(resolutions=DEFAULT_RESOLUTIONS)

    # Example files to visualise (one path per resolution)
    csv_paths = [
        "brownian_motion/bm_path_1_res_5.csv",
        "brownian_motion/bm_path_1_res_11.csv",
        "brownian_motion/bm_path_1_res_21.csv",
        "brownian_motion/bm_path_1_res_101.csv",
    ]

    visualize_brownian_paths(csv_paths)

# Poisson Temporal Point Process

In [ ]:
#!/usr/bin/env python3
"""
Time-discretised Poisson Point Process (TPP) workflow
====================================================

  1.  Simulate base data at λ = 1.0       →  tpp_data.csv
  2.  Draw N posterior-noise samples      →  tpp_noise_posterior_*.csv
  3.  Visualise *original* sequence       →  tpp_original_events.*
  4.  Re-evaluate samples at λ* values    →  tpp_counterfactual_rate<tag>_events.*

Hyper-parameters
----------------
• POSTERIOR_RATES  — space-separated list of alternative λ values treated as
  *counterfactual* intensities; defaults to 1.2 and 0.8.

Colours are now defined **explicitly** rather than via palette indices:

```text
ORIGINAL            → muted[0]
COUNTERFACTUAL 1.2  → muted[3]
COUNTERFACTUAL 0.8  → muted[5]
```

Author :  <your-name>
Updated:  29-Apr-2025
"""
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ---------------------------------------------------------------------
# Global styling & explicit colours
# ---------------------------------------------------------------------
sns.set_theme(style="white", palette="muted")
PALETTE = sns.color_palette("muted")

COLOR_ORIGINAL: tuple[float, float, float] = PALETTE[0]
COLOR_COUNTERFACTUAL: dict[float, tuple[float, float, float]] = {
    1.2: PALETTE[3],
    0.8: PALETTE[2],
}


# ---------------------------------------------------------------------
# Configuration (defaults overridable from CLI)
# ---------------------------------------------------------------------
TIME_HORIZON   = 10.0           # total time T
NUM_STEPS      = 100            # number of discretised intervals
RATE_BASE      = 1.0            # λ for the primary data
POSTERIOR_RATES_DEFAULT = [1.2, 0.8]
N_POST_SAMPLES = 5              # posterior noise draws
OUTPUT_FOLDER  = "pp_counterfactual"
SEED           = 42             # RNG seed

# Helper to slugify λ values such that 0.8 → "0p8" for filenames
_tag_from_rate = lambda r: re.sub(r"[.]+", "p", f"{r}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def create_folder(folder: str | Path = OUTPUT_FOLDER) -> None:
    Path(folder).mkdir(parents=True, exist_ok=True)


def _threshold(rate: float, dt: float) -> float:
    """Probability of ≥1 event in an interval Δt under homogeneous PPP."""
    return 1.0 - np.exp(-rate * dt)


# ── 1. Simulation ────────────────────────────────────────────────────

def simulate_tpp(
    time_horizon: float = TIME_HORIZON,
    num_steps: int = NUM_STEPS,
    rate: float = RATE_BASE,
    seed: int | None = None,
) -> pd.DataFrame:
    """Simulate a time-discretised Poisson point process at intensity *rate*."""
    rng   = np.random.default_rng(seed)
    times = np.linspace(0.0, time_horizon, num_steps + 1)       # grid t₀ … t_N
    dt    = times[1] - times[0]
    u     = rng.uniform(0.0, 1.0, size=num_steps)

    event = (u < _threshold(rate, dt)).astype(int)

    df = pd.DataFrame({"time": times[:-1], "u": u, "event": event})
    df.attrs["threshold"] = _threshold(rate, dt)
    df.attrs["dt"]        = dt
    return df


def save_dataframe(df: pd.DataFrame, path: str | Path) -> None:
    df.to_csv(path, index=False)
    print(f"Saved → {path}")


# ── 2. Posterior noise resampling ────────────────────────────────────

def draw_noise_posterior_samples(
    df: pd.DataFrame,
    n_samples: int = N_POST_SAMPLES,
    output_folder: str | Path = OUTPUT_FOLDER,
    seed: int | None = None,
) -> None:
    """Generate *n_samples* noise vectors u′ conditioned on the event field."""
    rng       = np.random.default_rng(seed)
    threshold = df.attrs["threshold"]

    for s in range(1, n_samples + 1):
        new_u = np.where(
            df["event"].values == 1,
            rng.uniform(0.0, threshold, size=len(df)),
            rng.uniform(threshold, 1.0,   size=len(df)),
        )
        df_post       = df.copy()
        df_post["u"]  = new_u
        save_dataframe(df_post, Path(output_folder) / f"tpp_noise_posterior_{s}.csv")


# ── 3. Styling utilities ─────────────────────────────────────────────

def _apply_reference_style(ax, title: str, ylabel: str | None = None):
    ax.set_title(title, fontsize=28)
    ax.set_xlabel("Time", fontsize=24)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=24)
    ax.tick_params(axis="both", labelsize=20)
    sns.despine(left=True, bottom=False)
    ax.grid(False)


# ── 4. Plot: ORIGINAL sequence ───────────────────────────────────────

def visualise_original_sequence(csv_path: str | Path, output_folder: str | Path = OUTPUT_FOLDER):
    df  = pd.read_csv(csv_path)
    dt  = df.attrs.get("dt", TIME_HORIZON / NUM_STEPS)
    Δt2 = 0.5 * dt

    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
    event_times = df.loc[df["event"] == 1, "time"].to_numpy()

    ax.scatter(event_times + Δt2,
               np.ones_like(event_times),
               marker="|", s=400, linewidths=3,
               color=COLOR_ORIGINAL, alpha=0.9)

    ax.set_xlim(0, TIME_HORIZON)
    ax.set_ylim(0.5, 1.5)
    ax.set_yticks([1])
    ax.set_yticklabels(["Sequence"])

    _apply_reference_style(ax, "Original Sequence (λ = 1)")
    fig.tight_layout()

    jpg = Path(output_folder) / "tpp_original_events.jpg"
    pdf = Path(output_folder) / "tpp_original_events.pdf"
    fig.savefig(jpg, bbox_inches="tight");  fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"Figure saved → {jpg}  &  {pdf}")


# ── 5. Plot: COUNTERFACTUAL sequences ────────────────────────────────

def visualise_counterfactual_rate(
    posterior_paths: list[str | Path],
    rate: float,
    output_tag: str,
    output_folder: str | Path = OUTPUT_FOLDER,
):
    palette = PALETTE  # shorthand

    # Determine the colour to use for *all* samples at this rate
    base_colour = COLOR_COUNTERFACTUAL.get(rate)

    # Infer dt from first file (shared grid)
    base_df = pd.read_csv(posterior_paths[0])
    dt      = TIME_HORIZON / NUM_STEPS if "dt" not in base_df.attrs else base_df.attrs["dt"]
    Δt2     = 0.5 * dt
    thr_alt = _threshold(rate, dt)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

    for idx, path in enumerate(sorted(posterior_paths, key=str)):
        df          = pd.read_csv(path)
        events_alt  = df["u"].values < thr_alt
        event_times = df.loc[events_alt, "time"].to_numpy()

        colour = base_colour if base_colour is not None else palette[idx % len(palette)]
        y_level = idx + 1
        ax.scatter(event_times + Δt2,
                   np.full_like(event_times, y_level),
                   marker="|", s=400, linewidths=3,
                   color=colour, alpha=0.9)

    ax.set_xlim(0, TIME_HORIZON)
    ax.set_ylim(0.5, len(posterior_paths) + 0.5)
    ax.set_yticks(range(1, len(posterior_paths) + 1))
    ax.set_yticklabels([f"Sample {s}" for s in range(1, len(posterior_paths) + 1)])

    _apply_reference_style(ax, f"Counterfactual Sequence (λ = {rate})")
    fig.tight_layout()

    jpg = Path(output_folder) / f"tpp_counterfactual_rate{output_tag}_events.jpg"
    pdf = Path(output_folder) / f"tpp_counterfactual_rate{output_tag}_events.pdf"
    fig.savefig(jpg, bbox_inches="tight");  fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"Figure saved → {jpg}  &  {pdf}")


# ---------------------------------------------------------------------
# Main program
# ---------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="TPP workflow with explicit colours")
    parser.add_argument(
        "--posterior-rates", "-r", metavar="λ", type=float, nargs="+",
        default=POSTERIOR_RATES_DEFAULT,
        help="Space-separated λ values for counterfactual evaluation (default: 1.2 0.8)",
    )

    # Ignore stray Jupyter/Colab flags like -f <kernel.json>
    args, _ = parser.parse_known_args()
    posterior_rates: list[float] = args.posterior_rates

    create_folder()

    # 1 ▶ Simulate & save base data (λ = 1.0)
    base_df      = simulate_tpp(seed=SEED)
    base_csvpath = Path(OUTPUT_FOLDER) / "tpp_data.csv"
    save_dataframe(base_df, base_csvpath)

    # 2 ▶ Draw posterior-noise samples
    draw_noise_posterior_samples(base_df, seed=SEED + 1)

    # 3 ▶ Plot original sequence
    visualise_original_sequence(base_csvpath)

    # Common list of posterior files
    posterior_files = [Path(OUTPUT_FOLDER) / f"tpp_noise_posterior_{s}.csv" for s in range(1, N_POST_SAMPLES + 1)]

    # 4 ▶ Counterfactual plots for each λ*
    for rate in posterior_rates:
        tag = _tag_from_rate(rate)
        visualise_counterfactual_rate(posterior_files, rate=rate, output_tag=tag)


if __name__ == "__main__":
    main()


# Spatio-temporal Point Process

In [ ]:
# --------------------  Colab CELL 1  --------------------
# (1) simulate + (2) derive posterior noises + (3) counterfactual runs

import numpy as np
import os
import pandas as pd

# ---------- 1. GLOBAL SETTINGS ----------
SEED = 42
np.random.seed(SEED)

# Grid: 100×100 in space, 1000 in time  →  Δx = Δy = 0.01, Δt = 0.01
NX, NY, NT = 100, 100, 1000
DELTA_T = 10.0 / NT
DELTA_S = (1.0 / NX) * (1.0 / NY)          # area of one pixel

# Hawkes-RBF kernel parameters
mu     = 1.0          # base rate
alpha  = 7.0          # branching factor
tau    = 1.0          # temporal scale (sec)
sigma  = 0.02         # spatial scale (unit square)

mu_cf  = 2.0          # new background rate for counterfactual step

# ---------- 2. PRE-ALLOCATE TENSORS ----------
rand_tensor       = np.random.rand(NX, NY, NT).astype(np.float32)
event_tensor      = np.zeros_like(rand_tensor, dtype=np.uint8)
threshold_tensor  = np.zeros_like(rand_tensor, dtype=np.float32)

# coordinates for vectorised distance computations
xs = np.linspace(0.5/NX, 1 - 0.5/NX, NX)      # cell centres
ys = np.linspace(0.5/NY, 1 - 0.5/NY, NY)
X, Y = np.meshgrid(xs, ys, indexing='ij')

# store history as lists for efficiency
history_times, history_xs, history_ys = [], [], []

def rbf_kernel(dt, dx2):
    """α * exp(-dt/τ) * exp(-dx2 / (2σ²))   (dt > 0 assumed)"""
    return alpha * np.exp(-dt / tau) * np.exp(-dx2 / (2.0 * sigma**2))

# ---------- 3. MAIN SIMULATION LOOP ----------
for t_idx in range(NT):
    t_curr = t_idx * DELTA_T
    # start with constant background
    intensity = np.full((NX, NY), mu, dtype=np.float32)

    # add contributions from previous events
    if history_times:                         # skip empty history
        dt = t_curr - np.asarray(history_times)          # shape (n_events,)
        w_t = np.exp(-dt / tau)                          # temporal decay  (n_events,)
        for ev_idx, (hx, hy, wt) in enumerate(zip(history_xs, history_ys, w_t)):
            dx2 = (X - hx)**2 + (Y - hy)**2              # (NX,NY)
            intensity += alpha * wt * np.exp(-dx2 / (2.0 * sigma**2))

    # convert to threshold   τ = λ Δt Δs
    thresh = intensity * DELTA_T * DELTA_S
    threshold_tensor[..., t_idx] = thresh

    # decide events
    new_events = rand_tensor[..., t_idx] < thresh
    event_tensor[..., t_idx] = new_events.astype(np.uint8)

    # append new events to history
    if new_events.any():
        new_x, new_y = np.where(new_events)
        history_times.extend([t_curr] * len(new_x))
        history_xs.extend(xs[new_x])
        history_ys.extend(ys[new_y])

print(f"Simulation finished with {event_tensor.sum()} events.")

# ---------- 4. NOISE-POSTERIOR SAMPLING ----------
noise_tensors = []
for _ in range(3):
    noise = np.empty_like(rand_tensor, dtype=np.float32)
    # where event ==1  → sample U(0, τ)  else U(τ,1)
    mask = event_tensor.astype(bool)
    noise[mask] = np.random.rand(mask.sum()).astype(np.float32) * threshold_tensor[mask]
    noise[~mask] = threshold_tensor[~mask] + np.random.rand((~mask).sum()).astype(np.float32) * (1.0 - threshold_tensor[~mask])
    noise_tensors.append(noise)

# ---------- 5. COUNTERFACTUAL SIMULATIONS ----------
def simulate_given_noise(mu_val, noise_tensor):
    """Re-run event generation with a new μ but identical uniform noise."""
    cf_events = np.zeros_like(noise_tensor, dtype=np.uint8)
    hist_t, hist_x, hist_y = [], [], []
    for t_idx in range(NT):
        t_curr  = t_idx * DELTA_T
        intensity = np.full((NX, NY), mu_val, dtype=np.float32)
        if hist_t:
            dt = t_curr - np.asarray(hist_t)
            w_t = np.exp(-dt / tau)
            for hx, hy, wt in zip(hist_x, hist_y, w_t):
                dx2 = (X - hx)**2 + (Y - hy)**2
                intensity += alpha * wt * np.exp(-dx2 / (2.0 * sigma**2))
        thresh = intensity * DELTA_T * DELTA_S
        new_ev = noise_tensor[..., t_idx] < thresh
        cf_events[..., t_idx] = new_ev.astype(np.uint8)
        if new_ev.any():
            nx, ny = np.where(new_ev)
            hist_t.extend([t_curr]*len(nx))
            hist_x.extend(xs[nx])
            hist_y.extend(ys[ny])
    return cf_events

cf_event_tensors = [simulate_given_noise(mu_cf, rand_tensor)] + \
                   [simulate_given_noise(mu_cf, n) for n in noise_tensors]

# ---------- 6. SAVE EVERYTHING ----------
os.makedirs("stpp_model", exist_ok=True)
pd.DataFrame(rand_tensor.reshape(-1)).to_csv("stpp_model/random_tensor.csv", index=False, header=False)
pd.DataFrame(event_tensor.reshape(-1)).to_csv("stpp_model/event_tensor.csv", index=False, header=False)
pd.DataFrame(threshold_tensor.reshape(-1)).to_csv("stpp_model/threshold_tensor.csv", index=False, header=False)

for i, noise in enumerate(noise_tensors, 1):
    pd.DataFrame(noise.reshape(-1)).to_csv(f"stpp_model/noise_tensor_{i}.csv", index=False, header=False)

for i, cf in enumerate(cf_event_tensors):
    pd.DataFrame(cf.reshape(-1)).to_csv(f"stpp_model/cf_event_tensor_{i}.csv", index=False, header=False)

print("All tensors saved to  stpp_model/  (flat CSV, one value per line).")





# --------------------  Colab CELL 2 (hollow markers) --------------------
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

palette = sns.color_palette("muted")


# ----------- constants (same as in Cell 1) -----------
NX, NY, NT   = 100, 100, 1000
DELTA_T      = 10.0 / NT
xs = np.linspace(0.5/NX, 1 - 0.5/NX, NX)
ys = np.linspace(0.5/NY, 1 - 0.5/NY, NY)

def load_tensor(path):
    flat = pd.read_csv(path, header=None).values.flatten()
    return flat.reshape(NX, NY, NT)

orig = load_tensor("stpp_model/event_tensor.csv")
cfs  = [load_tensor(f"stpp_model/cf_event_tensor_{i}.csv") for i in range(4)]

# seaborn-muted palette colours
BLUE = palette[0]
RED  = palette[3]

def scatter_base(ax, E, marker, edge_col):
    """Scatter events with hollow markers (transparent face)."""
    ix, iy, it = np.where(E == 1)
    if len(ix) == 0:
        return
    x = xs[ix];  y = ys[iy];  t = it * DELTA_T
    # encode time in size (older → smaller)
    ages  = (t.max() - t) / t.max()
    sizes = 10 + 120 * (1 - ages)
    ax.scatter(
        x, y,
        s=sizes,
        facecolors='none',            # transparent fill
        edgecolors=edge_col,          # coloured border
        linewidths=1.2,
        marker=marker
    )

def scatter_panel(ax, E, title, marker='o', edge_col=BLUE):
    scatter_base(ax, E, marker, edge_col)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)

def scatter_cf(ax, cf, title):
    shared = (cf == 1) & (orig == 1)
    new    = (cf == 1) & (orig == 0)
    scatter_base(ax, shared.astype(np.uint8), marker='o', edge_col=RED)
    scatter_base(ax, new.astype(np.uint8),    marker='s', edge_col=RED)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)

# ----------  PLOTTING ----------
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
scatter_panel(axes[0], orig, "Original (μ = 1)", marker='o', edge_col=BLUE)

for i, cf in enumerate(cfs, 1):
    scatter_cf(axes[i], cf, f"Four Counterfactuals (μ = 2)")

# Legend with hollow markers
handles = [
    Line2D([0], [0], marker='o', markersize=8, markerfacecolor='none',
           markeredgewidth=1.2, markeredgecolor=BLUE, linestyle='None',
           label="Original event"),
    Line2D([0], [0], marker='o', markersize=8, markerfacecolor='none',
           markeredgewidth=1.2, markeredgecolor=RED, linestyle='None',
           label="Event present in both"),
    Line2D([0], [0], marker='s', markersize=8, markerfacecolor='none',
           markeredgewidth=1.2, markeredgecolor=RED, linestyle='None',
           label="New event (μ↑)"),
]

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=11)

fig.suptitle(
    "Spatio-temporal Hawkes simulations — effect of doubling μ\n"
    "Blue outline = observed •  Red circle = shared •  Red square = brand-new",
    y=1.08, fontsize=14
)

fig.subplots_adjust(top=0.82, right=0.8, wspace=0.05)
plt.savefig("stpp_model/event_scatters.pdf", bbox_inches='tight')
plt.show()
print("Figure saved to  stpp_model/event_scatters.pdf")


# Spatio-Temporal Forest Conflict

## Simulation

(Adopt parameters in "2. Experiment hyper-parameters " when you want to replicate the paper). We used the plots *mining_off_J20_run13_density_events* and *mining_on_J100_run47_rain_events*.

In [ ]:
# ================================================================
#  Forest–Mining–Conflict–Rain Simulator  (GP-rain version, 2025-07-01)
#  --------------------------------------------------------------
#  • 2 mining-setups × 3 J values × NUM_RUNS (cached to disk)
#  • A *single* resolution parameter **J** now determines
#    – the number of time steps **T ≔ J**
#    – the spatial grid size **J × J**
#  • Saves full tensors {"rho","mines","conflicts","rain"} for
#    every run so the visualiser can open them.
# ================================================================
from __future__ import annotations

from pathlib import Path
from itertools import product
import pickle
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── extra imports needed for the GP rain model ───────────────────
from scipy.spatial.distance import cdist
from scipy.linalg import cholesky

# ---------- 0. Progress-bar utility -----------------------------------------
try:
    from tqdm.notebook import tqdm            # nicer in Jupyter / Colab
except Exception:                             # noqa: BLE001
    try:
        from tqdm.auto import tqdm            # works elsewhere
    except Exception:                         # noqa: BLE001
        tqdm = None                           # type: ignore


def get_pbar(total: int, desc: str):
    """Return a context-managed progress-bar or a dummy printer."""

    class _Dummy:
        def __enter__(self):                # noqa: D401
            print(f"[{desc}] start ({total} steps)")
            self.i = 0
            return self

        def set_postfix(self, **kwargs):    # noqa: D401
            if kwargs:
                kv = ", ".join(f"{k}={v}" for k, v in kwargs.items())
                print(f" → {kv}")

        def update(self, n=1):
            self.i += n
            print(f"   {self.i}/{total}")

        def __exit__(self, exc_type, exc, tb):  # noqa: D401
            print(f"[{desc}] done")

    return tqdm(total=total, desc=desc) if tqdm else _Dummy()


# ---------- 1. Environment-aware output directory ---------------------------
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

SIM_DIR = Path("/content/drive/MyDrive/B_ForestConflict") if IN_COLAB else Path("ForestConflict")
SIM_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 2. Experiment hyper-parameters ----------------------------------
NUM_RUNS = 10 # Paper EXP runs are made with 200
J_LIST = [20, 50]#, 75, 100]            # Paper EXP runs are made with [20, 50, 75, 100]
SETUPS = {
    "mining_on": 10.0,
    "mining_off": 5.0,
}

# Stable setup-specific seed offsets.
# This replaces Python's built-in hash(...), which is randomized across sessions.
SETUP_SEED_OFFSETS = {
    "mining_on": 0,
    "mining_off": 10_000,
}

# ---------- 3. Model parameters (unchanged) ---------------------------------
#
# Rule of thumb: ↑ value = “more / bigger / stronger / slower to change”
# but at the price of higher run-time if it affects grid size J or N.
# ----------------------------------------------------------------------------

L = 1.0            # Physical side-length of the square domain.

RAIN_PARAMS = {
    "threshold": -1.7,    # GP value below which a cell is flagged “rain”.
                           # ↑threshold (toward 0) ⇒ rain is more frequent;
                           # ↓threshold ⇒ sparser rain events.

    "growth_mult": 2.0,    # Factor multiplying tree growth *when raining*.
                           # ↑growth_mult ⇒ rain benefits vegetation more.

    "lengthscale": 0.10,   # Spatial GP length-scale (in domain units).
                           # ↑lengthscale ⇒ broader, smoother rain patches;
                           # ↓lengthscale ⇒ smaller, speckled showers.

    "persist": 0.90,       # Interpreted below as the correlation over ONE
                           # full unit of simulation time (not per discrete step).
                           # This is important: the old code used a fixed AR(1)
                           # coefficient per step, which changes meaning when J
                           # changes. We now convert this into an OU rate so that
                           # refinement J→∞ approximates the same continuous-time
                           # process.
}

DENSITY_PARAMS = {
    "r": 1.2,      # Intrinsic logistic growth rate of trees.
                   # ↑r ⇒ faster rebound toward carrying capacity.

    "D": 5e-4,     # Diffusion coefficient (Laplacian term).
                   # ↑D ⇒ tree density smooths out more quickly across space.
}

MINING_PARAMS = {
    "radius_sq": 0.05**2,   # Radius² cleared by each mine site.
                            # ↑radius_sq ⇒ mines devastate a larger footprint.
}

CONFLICT_PARAMS = {
    "base_rate": 100.0,          # Baseline Poisson intensity of conflict
                                 # around active mines. ↑base_rate ⇒ more
                                 # conflicts per timestep per km².

    "interaction_radius_sq": 0.2**2,  # How far a mine’s influence extends.
                                      # ↑value ⇒ conflicts can ignite further
                                      # from the mine.

    "time_window": 0.20,          # Look-back window (simulation time units)
                                   # when checking for recent mines/conflicts.
                                   # ↑time_window ⇒ longer social memory.
}


# ---------- 4. Grid & helper utilities (unchanged except for J) -------------
#
# A helper to (re)initialise global grid variables whenever J changes.
# ----------------------------------------------------------------------------
def _set_grid(J: int):
    """Update global grid variables for the current resolution J."""
    global N, x, y, X, Y, dx
    N = J
    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)
    X, Y = np.meshgrid(x, y, indexing="ij")
    dx = 1 / (N - 1)


# Initialise with a default J so the helper functions are defined safely.
_set_grid(J_LIST[0])


def laplacian_cell(Z: np.ndarray, j: int, k: int) -> float:
    up, down = Z[j - 1 if j else j, k], Z[j + 1 if j < N - 1 else j, k]
    left, right = Z[j, k - 1 if k else k], Z[j, k + 1 if k < N - 1 else k]
    return (up + down + left + right - 4.0 * Z[j, k]) / dx**2


def spatial_smooth(field: np.ndarray, steps: int) -> np.ndarray:
    for _ in range(steps):
        field = (field +
                 np.roll(field, 1, 0) + np.roll(field, -1, 0) +
                 np.roll(field, 1, 1) + np.roll(field, -1, 1)) / 5.0
    return field


# ---------- 5. Initialisation functions (unchanged except for J) ------------
def init_tree_density() -> np.ndarray:
    rho0 = (0.6
            + 0.3 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y)
            - 0.7 * np.exp(-((X - 0.25) ** 2 + (Y - 0.70) ** 2) / 0.015)
            + 0.25 * np.exp(-((X - 0.80) ** 2 + (Y - 0.30) ** 2) / 0.005))
    return rho0.clip(0, 1)


def init_mining_events() -> np.ndarray:
    return np.zeros((N, N), dtype=np.uint8)


def init_conflict_events() -> np.ndarray:
    return np.zeros((N, N), dtype=np.uint8)


# ---------- 6. OU-in-time + spatial GP helper for rain ----------------------
def ou_gp_3d(N_spatial: int, N_temporal: int, *,
             dt: float,
             spatial_lengthscale: float = 0.1,
             temporal_correlation: float = 0.8,
             variance: float = 1.0,
             seed: int | None = None) -> np.ndarray:
    """
    Return a (T, N, N) array from a continuous-time Ornstein–Uhlenbeck (OU)
    process in time, with a squared-exponential GP in space.

    Continuous-time model:
        dX_t(s) = -lambda * X_t(s) dt + sigma * dW_t^(K)(s),

    where:
      • s is the spatial location,
      • W_t^(K) is Gaussian noise that is white in time but correlated in space
        with covariance kernel K,
      • the stationary marginal field at each fixed time has covariance
            variance * K.

    Why this is better than a fixed AR(1) coefficient:
      If one keeps X_t = rho * X_{t-1} + ... with rho fixed while dt changes,
      then the physical time-correlation changes with resolution J. That breaks
      meaningful convergence as J→∞.

      Here, we instead derive the discrete update from the OU process:
          X_{n+1} = a * X_n + eta_n,
      with
          a = exp(-lambda * dt),
      and eta_n chosen so that the stationary marginal variance stays constant.

      We interpret `temporal_correlation` as the correlation over ONE unit of
      simulation time. Therefore, for any dt:
          a(dt) = temporal_correlation ** dt = exp(log(rho_unit) * dt)
      which is exactly resolution-consistent.
    """
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()

    # Build the spatial covariance matrix on the N_spatial x N_spatial grid.
    grid = np.linspace(0, 1, N_spatial)
    Xg, Yg = np.meshgrid(grid, grid, indexing="ij")
    coords = np.column_stack([Xg.ravel(), Yg.ravel()])
    dists = cdist(coords, coords)

    # Squared-exponential spatial covariance.
    K = variance * np.exp(-0.5 * (dists / spatial_lengthscale) ** 2)

    # Small jitter for numerical stability.
    K += 1e-6 * np.eye(K.shape[0])

    # Cholesky factor so Lc @ N(0, I) ~ N(0, K)
    Lc = cholesky(K, lower=True)

    field = np.empty((N_temporal, N_spatial, N_spatial), dtype=np.float32)

    rho_unit = temporal_correlation
    if not (0.0 < rho_unit < 1.0):
        raise ValueError("temporal_correlation must lie in (0, 1)")

    # Resolution-consistent one-step persistence.
    a = rho_unit ** dt

    # Innovation scale chosen so stationary marginal covariance remains K.
    innovation_scale = np.sqrt(1.0 - a ** 2)

    # Initial state drawn from the stationary marginal GP(0, K).
    z0 = Lc @ rng.standard_normal(N_spatial * N_spatial)
    field[0] = z0.reshape(N_spatial, N_spatial)

    # OU recursion through time.
    for t_idx in range(1, N_temporal):
        epsilon = Lc @ rng.standard_normal(N_spatial * N_spatial)
        epsilon = epsilon.reshape(N_spatial, N_spatial)
        field[t_idx] = a * field[t_idx - 1] + innovation_scale * epsilon

    return field


# ---------- 7. Forward-in-time kernels (unchanged) --------------------------
def density_forward(rho_prev: np.ndarray, rain_prev: np.ndarray,
                    mines_prev: np.ndarray, dt: float) -> np.ndarray:
    rho_next = np.empty_like(rho_prev)
    r, D = DENSITY_PARAMS["r"], DENSITY_PARAMS["D"]
    for j in range(N):
        for k in range(N):
            growth_factor = r * (RAIN_PARAMS["growth_mult"]
                                 if rain_prev[j, k] else 1.0)
            ρp = rho_prev[j, k]
            g = growth_factor * ρp * (1 - ρp)
            lap = D * laplacian_cell(rho_prev, j, k)
            ρn = np.clip(ρp + dt * (g + lap), 0, 1)

            if mines_prev.any():
                for jm, km in np.argwhere(mines_prev == 1):
                    if ((x[j] - x[jm]) ** 2 + (y[k] - y[km]) ** 2) \
                            <= MINING_PARAMS["radius_sq"]:
                        ρn = 0.0
                        break
            rho_next[j, k] = ρn
    return rho_next


def in_neighbourhood(arr: np.ndarray, i_ref: int, j: int, k: int, win: int) -> bool:
    for τ in range(max(0, i_ref - win), i_ref + 1):
        if not arr[τ].any():
            continue
        for jm, km in np.argwhere(arr[τ] == 1):
            if ((x[j] - x[jm]) ** 2 + (y[k] - y[km]) ** 2) \
                    <= CONFLICT_PARAMS["interaction_radius_sq"]:
                return True
    return False


def mining_forward(rng: np.random.Generator, rho_prev: np.ndarray,
                   conflict_prev: np.ndarray, intensity: float,
                   dt: float, win: int) -> np.ndarray:
    mines = np.zeros((N, N), dtype=np.uint8)
    for j in range(N):
        for k in range(N):
            λ = 0.0 if in_neighbourhood(conflict_prev,
                                        conflict_prev.shape[0] - 1,
                                        j, k, win) \
                else intensity * rho_prev[j, k]
            mines[j, k] = rng.random() <= λ * dx * dx * dt
    return mines


def conflict_forward(rng: np.random.Generator, mines_prev: np.ndarray,
                     conflict_prev: np.ndarray, dt: float, win: int) -> np.ndarray:
    conflicts = np.zeros((N, N), dtype=np.uint8)
    for j in range(N):
        for k in range(N):
            λc = (CONFLICT_PARAMS["base_rate"]
                  if in_neighbourhood(mines_prev,
                                      mines_prev.shape[0] - 1,
                                      j, k, win)
                  else 0.0)
            conflicts[j, k] = rng.random() <= λc * dx * dx * dt
    return conflicts


# ---------- 8. Single-simulation routine (OU rain integrated) ---------------
def run_once(J: int, intensity: float, seed: int) -> dict[str, np.ndarray]:
    _set_grid(J)                 # ensure grid matches this J

    # Use separate derived seeds for independent random streams.
    # This keeps the whole run reproducible, while avoiding reuse of the exact
    # same seed for both the rain-field generator and the event generator.
    rain_seed = seed + 1
    event_seed = seed + 2

    rng = np.random.default_rng(event_seed)
    dt = 1 / (J - 1)
    win = int(np.ceil(CONFLICT_PARAMS["time_window"] / dt))

    # Draw the latent rain field from the OU-in-time + spatial GP model.
    gp_field = ou_gp_3d(
        N_spatial=J,
        N_temporal=J,
        dt=dt,
        spatial_lengthscale=RAIN_PARAMS["lengthscale"],
        temporal_correlation=RAIN_PARAMS["persist"],
        variance=1.0,
        seed=rain_seed,
    ).astype(np.float32)

    # Threshold latent GP values into a binary rain mask.
    rain = (gp_field < RAIN_PARAMS["threshold"]).astype(np.uint8)

    rho       = np.empty((J, N, N), dtype=np.float32)
    mines     = np.zeros((J, N, N), dtype=np.uint8)
    conflicts = np.zeros((J, N, N), dtype=np.uint8)

    rho[0] = init_tree_density()
    mines[0] = init_mining_events()
    conflicts[0] = init_conflict_events()

    for i in range(1, J):
        rho[i] = density_forward(rho[i - 1], rain[i - 1], mines[i - 1], dt)
        mines[i] = mining_forward(rng, rho[i - 1], conflicts[:i], intensity, dt, win)
        conflicts[i] = conflict_forward(rng, mines[:i], conflicts[:i], dt, win)

    return {
        "rho": rho,
        "mines": mines,
        "conflicts": conflicts,
        "rain": rain,
    }


# ---------- 9. Full experiment (unchanged apart from stable seeds) ----------
def main():  # noqa: C901
    tasks = list(product(SETUPS.items(), J_LIST, range(NUM_RUNS)))

    existing_files: list[str] = []
    missing_tasks: list[tuple[str, int, int]] = []

    for (setup_name, _), J, run in tasks:
        sim_file = SIM_DIR / f"{setup_name}_J{J}_run{run}.pkl"
        if sim_file.exists():
            existing_files.append(sim_file.name)
        else:
            missing_tasks.append((setup_name, J, run))

    if existing_files:
        print(f"🗃 Found {len(existing_files)} cached simulations:")
        for f in sorted(existing_files):
            print("   ✅", f)
    print(f"🧪 Will run {len(missing_tasks)} new simulations:")
    for setup_name, J, run in missing_tasks:
        print(f"   🕒 {setup_name}_J{J}_run{run}.pkl")

    records: list[dict[str, int | str]] = []
    seed_base = 42

    with get_pbar(total=len(tasks), desc="Simulations") as pbar:
        for (setup_name, intensity), J, run in tasks:
            sim_file = SIM_DIR / f"{setup_name}_J{J}_run{run}.pkl"

            try:
                data = pickle.loads(sim_file.read_bytes()) if sim_file.exists() else None
                if not (isinstance(data, dict) and
                        all(k in data for k in ("rho", "mines", "conflicts", "rain"))):
                    raise ValueError("old-format pickle")
            except Exception:
                # Stable, reproducible seed across sessions / machines.
                setup_offset = SETUP_SEED_OFFSETS[setup_name]
                seed = seed_base + setup_offset + J * 100 + run

                data = run_once(J, intensity, seed)
                sim_file.write_bytes(pickle.dumps(data))

            conflict_total = int(data["conflicts"].sum())
            records.append({"setup": setup_name, "J": J,
                            "run": run, "conflicts": conflict_total})

            pbar.set_postfix(setup=setup_name, J=J, run=run,
                             conflicts=conflict_total)
            pbar.update(1)

    # ---------- 10. Save results ------------------------------------
    df = pd.DataFrame(records)
    df.to_csv("conflict_counts_grouped.csv", index=False)
    print("✅ Saved conflict_counts_grouped.csv")

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=df, x="J", y="conflicts",
                   hue="setup", palette="muted",
                   inner="quartile")
    plt.title(f"Total conflict events – {NUM_RUNS} runs per setup & J")
    plt.xlabel("Resolution (J)")
    plt.ylabel("Conflicts per run")
    plt.tight_layout()
    plt.savefig("conflict_violin_grouped.png", dpi=300)
    #plt.show()
    print("📊 Figure saved to conflict_violin_grouped.png")


if __name__ == "__main__":
    main()

## Plot Box

In [ ]:
import shutil
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
#  Adjust this if the variable already exists in your notebook / script
# ---------------------------------------------------------------------
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

SIM_DIR = Path("drive/MyDrive/B_ForestConflict") if IN_COLAB else Path("ForestConflict")
SIM_DIR.mkdir(parents=True, exist_ok=True)
print(SIM_DIR)

if IN_COLAB:
    src = Path("conflict_counts_grouped.csv")
    dst = Path("drive/MyDrive/B_ForestConflict/conflict_counts_grouped.csv")

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)

    print(f"📁 Copied {src} → {dst}")

# ---------------------------------------------------------------------
#  Main plotting helper
# ---------------------------------------------------------------------
def boxplot_from_csv():
    # ── 1. Global, publication-ready style ────────────────────────────
    sns.set_theme(style="whitegrid", palette="muted")
    mpl.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 14,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    # ── 2. Read CSV from the same simulation folder ───────────────────
    src_csv = SIM_DIR / "conflict_counts_grouped.csv"
    df = pd.read_csv(src_csv)
    print(f"📁 Loaded {src_csv}")

    # ── 3. Baseline box-plot (conflicts per setup) ────────────────────
    plt.figure(figsize=(5, 3.2))
    ax1 = sns.boxplot(data=df, x="J", y="conflicts",
                      hue="setup", palette="muted")

    ax1.set_title("Conflict Events by Setup")
    ax1.set_xlabel("Resolution (J)")
    ax1.set_ylabel("Conflicts per Run")

    for spine in ["left", "bottom"]:
        ax1.spines[spine].set_linewidth(1.5)
        ax1.spines[spine].set_color("black")

    ax1.yaxis.grid(True, linestyle=":", linewidth=0.7)
    ax1.xaxis.grid(False)

    ax1.legend(title="Setup", loc="center left",
               bbox_to_anchor=(1.02, 0.5), frameon=False)

    plt.tight_layout()
    plt.savefig(SIM_DIR / "conflict_boxplot.jpg", dpi=300, bbox_inches="tight")
    plt.savefig(SIM_DIR / "conflict_boxplot.pdf", bbox_inches="tight")
    print("🖼️ Saved conflict_boxplot.jpg / .pdf")
    plt.show()

    # ── 4. ATE (mining_on − mining_off) per run & J ───────────────────
    ate_records = []
    for J in df["J"].unique():
        pivot = (df[df["J"] == J]
                 .pivot(index="run", columns="setup", values="conflicts"))
        if {"mining_on", "mining_off"}.issubset(pivot.columns):
            diffs = pivot["mining_on"] - pivot["mining_off"]
            ate_records += [{"J": J, "ATE": d} for d in diffs]
        else:
            print(f"⚠️  Missing setup(s) for J={J}; skipped.")

    ate_df = pd.DataFrame(ate_records)

    plt.figure(figsize=(4.5, 3))
    ax2 = sns.boxplot(data=ate_df, x="J", y="ATE", color=sns.color_palette("muted")[3])
    if ax2.get_legend():
        ax2.get_legend().remove()

    ax2.axhline(0, color="black", linestyle="--", linewidth=1)
    ax2.set_title("ATE: Mining Reduction")
    ax2.set_xlabel("Resolution ($J$)")
    ax2.set_ylabel("Δ Conflicts")

    for spine in ["left", "bottom"]:
        ax2.spines[spine].set_linewidth(1.5)
        ax2.spines[spine].set_color("black")
    ax2.yaxis.grid(True, linestyle=":", linewidth=0.7)
    ax2.xaxis.grid(False)

    plt.tight_layout()
    plt.savefig(SIM_DIR / "ate_boxplot.jpg", dpi=300, bbox_inches="tight")
    plt.savefig(SIM_DIR / "ate_boxplot.pdf", bbox_inches="tight")
    print("🖼️ Saved ate_boxplot.jpg / .pdf")
    #plt.show()


# ---------------------------------------------------------------------
if __name__ == "__main__":
    boxplot_from_csv()

## Plot Full

In [ ]:
# ================================================================
#  Cell ― Batch visualisation of every simulation .pkl
#  ---------------------------------------------------------------
#  • For each run: draw 10×10 frames of (1) forest density+events
#    and (2) rainfall+events, save both PDF & JPG.
#  • Assumes each .pkl stores dict with keys
#      {"rho", "mines", "conflicts", "rain"}  (all T×N×N arrays).
#    Files that do not match are skipped.
# ================================================================
import math
import pickle
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

# ---------- 0. Locate (and ensure) simulation directory -----------
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

SIM_DIR = Path("/content/drive/MyDrive/B_ForestConflict") if IN_COLAB else Path("ForestConflict")
SIM_DIR.mkdir(parents=True, exist_ok=True)           # ← ensure it exists

OVERWRITE = True  # set False to skip already-visualised runs

# ---------- 1.  Helper: frame-grid plot ---------------------------
def _plot_frames(
    base_cube: np.ndarray,
    mines: np.ndarray,
    conflicts: np.ndarray,
    base_cmap: str,
    base_vmin: float,
    base_vmax: float,
    title_text: str,
    colorbar_label: str,
    out_path: Path,
) -> None:
    """Render grid of frames and save to both PDF and JPG."""
    T, N, _ = base_cube.shape
    n_cols = 10
    n_rows = math.ceil(T / n_cols)
    dx = 1 / (N - 1)

    plt.rcParams.update({"figure.figsize": (24, 24 * n_rows / 10),
                         "xtick.labelsize": 6, "ytick.labelsize": 6})

    fig, axes = plt.subplots(n_rows, n_cols, squeeze=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.02, top=0.92,
                        wspace=0.05, hspace=0.05)

    brown = sns.color_palette("muted")[5]  # mining ×,
    red = sns.color_palette("muted")[3]  # conflict ○

    for t in range(T):
        ax = axes[t // n_cols, t % n_cols]
        im = ax.imshow(base_cube[t],
                       vmin=base_vmin, vmax=base_vmax,
                       cmap=base_cmap, origin="lower",
                       extent=[0, 1, 0, 1])

        # mining events (brown ×)
        if mines[t].any():
            iy, ix = np.where(mines[t] == 1)
            ax.scatter(ix * dx, iy * dx, marker="x", s=260, alpha=0.7,
                       color="white", linewidth=8)
            ax.scatter(ix * dx, iy * dx, marker="x", s=250, alpha=1.0,
                       color=brown, linewidth=4)

        # conflict events (blue ○)
        if conflicts[t].any():
            iyc, ixc = np.where(conflicts[t] == 1)
            ax.scatter(ixc * dx, iyc * dx, marker="o", s=260, alpha=0.7,
                       facecolors="none", edgecolors="white", linewidth=8)
            ax.scatter(ixc * dx, iyc * dx, marker="o", s=250, alpha=1.0,
                       facecolors="none", edgecolors=red,  linewidth=4)

        ax.set_title(f"t = {t:02d}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    # hide unused panels
    for idx in range(T, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].axis("off")

    # colour-bar
    cbar_ax = fig.add_axes([0.93, 0.08, 0.015, 0.80])
    sm = cm.ScalarMappable(cmap=base_cmap,
                           norm=plt.Normalize(vmin=base_vmin, vmax=base_vmax))
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label=colorbar_label)

    fig.suptitle(title_text, y=0.97, fontsize=10)

    # --- save ----------------------------------------------------
    pdf_path = out_path.with_suffix(".pdf")
    jpg_path = out_path.with_suffix(".jpg")
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.savefig(jpg_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"✅  Saved {pdf_path.name} and {jpg_path.name}")

# ---------- 2.  Iterate over all .pkl files in random order -------
pkl_files = list(SIM_DIR.glob("*.pkl"))
random.shuffle(pkl_files)

if not pkl_files:
    raise FileNotFoundError(f"No .pkl files found in {SIM_DIR.resolve()}")

for pkl_path in pkl_files:
    try:
        data = pickle.loads(pkl_path.read_bytes())
    except Exception as e:
        print(f"⚠️  Could not load {pkl_path.name}: {e}")
        continue

    if not isinstance(data, dict) or not all(k in data for k in
                                             ("rho", "mines", "conflicts", "rain")):
        print(f"⤴️  {pkl_path.name} does not contain full tensors – skipped.")
        continue

    rho       = np.asarray(data["rho"])
    mines     = np.asarray(data["mines"])
    conflicts = np.asarray(data["conflicts"])
    rain      = np.asarray(data["rain"])

    # ---- figure 1: density + events ---------------------------------
    out_stem = pkl_path.with_suffix("").name + "_density_events"
    out_path = SIM_DIR / out_stem
    if not OVERWRITE and (out_path.with_suffix(".pdf").exists() and out_path.with_suffix(".jpg").exists()):
        print(f"⏭️  Skipped {out_path.name} (already exists)")
    else:
        _plot_frames(
            base_cube=rho,
            mines=mines,
            conflicts=conflicts,
            base_cmap="Greens",
            base_vmin=0, base_vmax=1,
            title_text=f"Forest density{pkl_path.name}",
            colorbar_label="Forest density ρ",
            out_path=out_path,
        )

    # ---- figure 2: rain + events ------------------------------------
    out_stem = pkl_path.with_suffix("").name + "_rain_events"
    out_path = SIM_DIR / out_stem
    if not OVERWRITE and (out_path.with_suffix(".pdf").exists() and out_path.with_suffix(".jpg").exists()):
        print(f"⏭️  Skipped {out_path.name} (already exists)")
    else:
        _plot_frames(
            base_cube=rain,
            mines=mines,
            conflicts=conflicts,
            base_cmap="Blues",
            base_vmin=0, base_vmax=1,
            title_text=f"Rain (blue){pkl_path.name}",
            colorbar_label="Rain (1 = yes)",
            out_path=out_path,
        )